<a href="https://colab.research.google.com/github/govind0429/AI_LAB/blob/main/WEEK8_FORWARD_BACKWARD_CHAINING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re

# --- 1. Fact Class ---
class Fact:
    def __init__(self, predicate, arguments):
        self.predicate = predicate
        self.arguments = tuple(arguments)

    def __eq__(self, other):
        return isinstance(other, Fact) and self.predicate == other.predicate and self.arguments == other.arguments

    def __hash__(self):
        return hash((self.predicate, self.arguments))

    def __repr__(self):
        return f"{self.predicate}({', '.join(self.arguments)})"


# --- 2. Rule Class ---
class Rule:
    def __init__(self, premises, conclusion):
        self.premises = premises
        self.conclusion = conclusion

    def __repr__(self):
        premise_str = " AND ".join(
            [f"{p}({', '.join(a)})" for p, a in self.premises]
        )
        return f"IF {premise_str} -> THEN {self.conclusion[0]}({', '.join(self.conclusion[1])})"


# --- 3. Input Normalization ---
def clean_text_to_standard(text):
    text = text.replace("∧", " AND ").replace("^", " AND ")

    text = re.sub(r'\band\b', ' AND ', text, flags=re.IGNORECASE)
    text = text.replace("→", " -> ")

    text = re.sub(r'\bTHEN\b', ' -> ', text, flags=re.IGNORECASE)
    text = re.sub(r'\bIF\b', '', text, flags=re.IGNORECASE)

    text = re.sub(r'\s+', ' ', text).strip()

    return text


# --- 4. Predicate Parsing ---
def parse_flexible_predicate(text, default_arg="x"):
    text = text.strip()

    # Case 1: Standard format: Fever(P) or Fever(x)
    match = re.match(r"^([A-Za-z0-9_]+)\s*\(([^)]*)\)$", text)

    if match:
        pred = match.group(1)
        args_str = match.group(2).strip()

        if not args_str:
            return pred, []

        args = [arg.strip() for arg in args_str.split(",")]
        return pred, args

    # Case 2: Natural English format
    clean_text = re.sub(
        r'\b(a|an|the|patient|has|have|signs|of)\b',
        '',
        text,
        flags=re.IGNORECASE
    )

    clean_text = "".join(
        c for c in clean_text if c.isalnum() or c.isspace()
    ).strip()

    words = clean_text.split()

    if not words:
        raise ValueError("Invalid predicate expression.")

    pred_name = "".join(word.capitalize() for word in words)

    return pred_name, [default_arg]


# --- 5. Matching a Single Premise ---
def match_premise(premise, kb_facts, substitution):
    p_pred, p_args = premise
    matches = []

    for fact in kb_facts:

        if fact.predicate != p_pred:
            continue

        if len(fact.arguments) != len(p_args):
            continue

        new_sub = substitution.copy()
        possible_match = True

        for p_arg, f_arg in zip(p_args, fact.arguments):

            if p_arg.islower():  # Variable

                if p_arg in new_sub and new_sub[p_arg] != f_arg:
                    possible_match = False
                    break

                new_sub[p_arg] = f_arg

            else:  # Constant

                if p_arg != f_arg:
                    possible_match = False
                    break

        if possible_match:
            matches.append(new_sub)

    return matches


# --- 6. Matching All Premises ---
def match_all_premises(premises, kb_facts, substitution=None):

    if substitution is None:
        substitution = {}

    if not premises:
        return [substitution]

    first = premises[0]
    rest = premises[1:]

    first_matches = match_premise(first, kb_facts, substitution)

    all_bindings = []

    for sub in first_matches:
        bindings = match_all_premises(rest, kb_facts, sub)
        all_bindings.extend(bindings)

    return all_bindings


# --- 7. Forward Chaining Algorithm ---
def fol_fc_ask(kb_rules, kb_facts, query_predicate):

    print("\n--- Running Forward Chaining Inference Engine ---")

    # Check whether goal already exists
    for fact in kb_facts:

        if fact.predicate.lower() == query_predicate.lower():
            print(f"Goal directly found: {fact}")
            return {"x": fact.arguments}

    # Continue until no new facts are generated
    while True:

        new_facts = set()

        for rule in kb_rules:

            substitutions = match_all_premises(
                rule.premises, kb_facts
            )

            for theta in substitutions:

                c_pred, c_args = rule.conclusion

                inferred_args = [
                    theta.get(arg, arg) for arg in c_args
                ]

                inferred_fact = Fact(c_pred, inferred_args)

                if inferred_fact not in kb_facts and inferred_fact not in new_facts:

                    print(f"Rule triggered! Derived: {inferred_fact}")

                    new_facts.add(inferred_fact)

                    if inferred_fact.predicate.lower() == query_predicate.lower():

                        kb_facts.update(new_facts)

                        print(
                            f"\nGoal Reached! Confirmed assertion '{query_predicate}'."
                        )

                        return theta

        if not new_facts:

            print("\nEnd of stream: No additional deductions available.")

            return False

        kb_facts.update(new_facts)


# --- 8. Main Function ---
def main():

    print("======================================================")
    print("   Multi-Format First-Order Logic Reasoning Engine")
    print("======================================================")
    print("Supports: English phrases, Math logic (^, ∧, ->) & Standard FOL\n")

    # Get individual entity
    patient_id = input(
        "Enter individual entity label (e.g., P, John): "
    ).strip()

    if not patient_id:
        patient_id = "P"

    # STEP 1: Collect Facts
    print("\n--- STEP 1: Enter Initial Facts ---")

    kb_facts = set()

    while True:

        inp = input("Add Fact: ").strip()

        if inp.lower() == "done":
            break

        if not inp:
            continue

        try:

            pred, args = parse_flexible_predicate(
                inp, default_arg=patient_id
            )

            if args == ["x"]:
                args = [patient_id]

            kb_facts.add(Fact(pred, args))

        except Exception as e:
            print(f"Parsing error: {e}")

    # STEP 2: Collect Rules
    print("\n--- STEP 2: Enter Inference Rules ---")
    print("Type 'done' when finished.")

    kb_rules = []

    while True:

        inp = input("Add Rule: ").strip()

        if inp.lower() == "done":
            break

        if not inp:
            continue

        try:

            normalized = clean_text_to_standard(inp)

            if "->" not in normalized:
                print("Error: Missing implication operator (->).")
                continue

            premise_part, conclusion_part = normalized.split("->", 1)

            premises_raw = premise_part.split(" AND ")

            premises = [
                parse_flexible_predicate(p, "x")
                for p in premises_raw if p.strip()
            ]

            conclusion = parse_flexible_predicate(
                conclusion_part, "x"
            )

            kb_rules.append(Rule(premises, conclusion))

        except Exception as e:
            print(f"Parsing error: {e}")

    # STEP 3: Collect Goal
    print("\n--- STEP 3: Define Target Goal Query ---")

    goal_raw = input(
        "What are you trying to prove? (e.g., TestFlu): "
    ).strip()

    goal_pred, _ = parse_flexible_predicate(goal_raw, "x")

    if not kb_facts or not kb_rules or not goal_pred:

        print("\nAborted. Knowledge base is incomplete.")

        return

    # Display Knowledge Base
    print("\n================ PARSED KNOWLEDGE BASE ================")

    print("Compiled Facts Base:", kb_facts)

    print("Compiled Rules Base:")

    for idx, rule in enumerate(kb_rules, 1):
        print(f"  [{idx}] {rule}")

    print(f"Goal State Target: Prove {goal_pred}({patient_id})")

    print("=======================================================")

    # STEP 4: Run Forward Chaining
    result = fol_fc_ask(
        kb_rules, kb_facts, goal_pred
    )

    print("\nFinal Engine Matrix Solution:", result)


# --- 9. Program Execution ---
if __name__ == "__main__":
    main()

   Multi-Format First-Order Logic Reasoning Engine
Supports: English phrases, Math logic (^, ∧, ->) & Standard FOL

Enter individual entity label (e.g., P, John): robert

--- STEP 1: Enter Initial Facts ---
Add Fact: American(Robert)
Add Fact: Enemy(A, America)
Add Fact:  Owns(A, T1)
Add Fact: Missile(T1)
Add Fact: done

--- STEP 2: Enter Inference Rules ---
Type 'done' when finished.
Add Rule:  American(p) ^ Weapon(q) ^ Sells(p, q, r) ^ Hostile(r) -> Criminal(p)
Add Rule: Missile(x) ^ Owns(A, x) -> Sells(Robert, x, A)
Add Rule:  Missile(x) -> Weapon(x)
Add Rule: Enemy(x, America) -> Hostile(x)
Add Rule: done

--- STEP 3: Define Target Goal Query ---
What are you trying to prove? (e.g., TestFlu): {American(Robert), Enemy(A, America), Owns(A, T1), Missile(T1)}

================ PARSED KNOWLEDGE BASE ================
Compiled Facts Base: {Enemy(A, America), American(Robert), Missile(T1), Owns(A, T1)}
Compiled Rules Base:
  [1] IF American(p) AND Weapon(q) AND Sells(p, q, r) AND Hostile(r